In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path(".")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Saved intermediate tables.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# Descriptive comparisons and inventor-country portfolios

Calculate CPC counts (SI S1), matched-patent differences (SI S2.4), and country portfolios for `6_viz_firm_cities.ipynb`. Run from the repository root after `1_ccmt_matching.ipynb`, with paths set in the first cell. The country section can be run independently of the two descriptive comparisons.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

CPC_INPUT = PATENT_TEXT / 'cpc_current.parquet'
PAIR_INPUT = BASELINE / 'tocheck.parquet'
APPLICATION_INPUT = PATENT_TEXT / 'application.tsv.zip'
INVENTOR_INPUT = PATENT_TEXT / 'patent_inventor.tsv.zip'
GEOCODE_INPUT = PATSTAT / 'geoc_inv.parquet'
OUT = OUTPUTS / 'descriptive'
OUT.mkdir(parents=True, exist_ok=True)


## 1. CPC codes per green and nongreen publication

Read `patent_id,subgroup_id` from PatentsView `cpc_current.parquet`. Green publications have a Y02/Y04 assignment. Count distinct ordinary (non-Y) codes at subclass, main-group and detailed levels. Subclasses are the first four characters; main groups are the text before `/`; detailed codes are full symbols.

Include every publication in the assignment table, including those with zero ordinary codes.


In [ ]:
publication_cpc = pd.read_parquet(CPC_INPUT, columns=['patent_id', 'subgroup_id'])
publication_cpc = publication_cpc.rename(columns={'subgroup_id': 'cpc'})
publication_cpc['patent_id'] = publication_cpc.patent_id.astype(str)
publication_cpc['cpc'] = publication_cpc.cpc.str.replace(' ', '', regex=False)
publication_cpc = publication_cpc.drop_duplicates(['patent_id', 'cpc'])
publication_cpc['green'] = publication_cpc.cpc.str.startswith(('Y02', 'Y04'))
population = publication_cpc.groupby('patent_id').green.any()
ordinary = publication_cpc[~publication_cpc.cpc.str.startswith('Y')].copy()
ordinary['subclass'] = ordinary.cpc.str[:4]
ordinary['main_group'] = ordinary.cpc.str.split('/').str[0]


In [ ]:
code_counts = ordinary.groupby('patent_id').agg(
    subclass=('subclass', 'nunique'),
    main_group=('main_group', 'nunique'),
    detailed=('cpc', 'nunique'),
).reindex(population.index, fill_value=0).join(population).reset_index()
code_counts.to_parquet(OUT / 'cpc_counts.parquet', index=False)
count_summary = code_counts.groupby('green')[['subclass', 'main_group', 'detailed']].agg(
    ['count', 'mean', 'std', 'median', 'min', 'max'])
count_summary.to_csv(OUT / 'cpc_count_summary.csv')
count_summary


In [ ]:
levels = [('subclass', 'Subclass'), ('main_group', 'Main group'), ('detailed', 'Detailed CPC')]
colors = ['#32789e', '#df842d']
fig, ax = plt.subplots(figsize=(6, 4))
for j, (level, label) in enumerate(levels):
    for green, offset, color in zip([False, True], [-0.18, 0.18], colors):
        values = code_counts.loc[code_counts.green == green, level]
        boxes = ax.boxplot(values.to_numpy(), positions=[j + offset], widths=0.32,
                           patch_artist=True, manage_ticks=False,
                           flierprops={'markersize': 2, 'alpha': 0.3})
        boxes['boxes'][0].set_facecolor(color)
ax.set_xticks(range(len(levels)), [label for _, label in levels])
ax.set_yscale('log')
ax.set_ylabel('Number of ordinary CPC codes')
ax.legend(handles=[Patch(facecolor=c, label=l) for c, l in zip(colors, ['Non-CCMT', 'CCMT'])])
fig.tight_layout()
fig.savefig(OUT / 'morecodes.pdf')
plt.show()


Zero counts are included in the exported counts and summaries. The logarithmic figure displays positive values.


## 2. Properties of matched patent pairs

Read `BASELINE/tocheck.parquet`, saved by `1_ccmt_matching.ipynb`, and retain pairs with text cosine distance `cos < 0.99`. The table contains publication IDs `ccmt_patent_id,patent_id` and family IDs `ccmt_docdb_family_id,docdb_family_id`.

Application dates use `patent_id,date`; inventor counts use distinct `inventor_id` values per publication. CPC counts come from the table above. Calculate each attribute difference over pairs with both values available.


In [ ]:
pairs = pd.read_parquet(PAIR_INPUT, columns=[
    'ccmt_patent_id', 'patent_id', 'ccmt_docdb_family_id', 'docdb_family_id', 'cos'])
pairs = pairs.loc[pairs.cos < 0.99].copy()
pairs[['ccmt_patent_id', 'patent_id']] = pairs[['ccmt_patent_id', 'patent_id']].astype(str)

applications = pd.read_csv(APPLICATION_INPUT, sep='	', dtype=str,
                           usecols=['patent_id', 'date'])
applications['application_year'] = applications.date.str[:4].astype(int)
applications = applications.loc[applications.application_year.between(1976, 2024),
                                ['patent_id', 'application_year']]
inventors = pd.read_csv(INVENTOR_INPUT, sep='	', dtype=str,
                       usecols=['patent_id', 'inventor_id'])
team_size = inventors.groupby('patent_id').inventor_id.nunique().rename('inventor_team').reset_index()


In [ ]:
def paired_difference(pair_ids, attributes, column):
    joined = pair_ids[['ccmt_patent_id', 'patent_id']].merge(
        attributes[['patent_id', column]], on='patent_id')
    joined = joined.merge(attributes[['patent_id', column]].rename(
        columns={'patent_id': 'ccmt_patent_id', column: 'green_value'}), on='ccmt_patent_id')
    return joined.green_value - joined[column]

differences = {
    'Application year': paired_difference(pairs, applications, 'application_year'),
    'Ordinary CPC codes': paired_difference(pairs, code_counts, 'detailed'),
    'Inventor team size': paired_difference(pairs, team_size, 'inventor_team'),
}
paired_summary = pd.DataFrame([
    {'property': name, 'n_pairs': values.count(), 'mean_difference': values.mean(),
     'standard_error': values.std(ddof=1) / np.sqrt(values.count())}
    for name, values in differences.items()
])
paired_summary.to_csv(OUT / 'paired_differences.csv', index=False)
paired_summary


Pair-level descriptive standard errors are `sd / sqrt(n)`.


## 3. Inventor-country family/CPC inputs

Join inventor locations (`appln_id,filing_date,ctry_code`) to the application/family and family/CPC tables.

Count location records by family, country and filing year, then divide by the family's total records in that year. Assign this country share to each detailed CPC. Save the full CPC symbol as `cpc` and the country weight as `wt` in `PATSTAT/cntry_inv_patfam_cpcall.parquet`.


In [ ]:
families = pd.read_parquet(PATSTAT / 'tls201.parquet',
                           columns=['appln_id', 'docdb_family_id']).drop_duplicates()
family_cpc = pd.read_parquet(PATSTAT / 'tls225.parquet',
                            columns=['docdb_family_id', 'cpc_class_symbol'])
family_cpc = family_cpc.rename(columns={'cpc_class_symbol': 'cpc'}).assign(ratio=1)
inventor_locations = pd.read_parquet(GEOCODE_INPUT,
                                    columns=['appln_id', 'filing_date', 'ctry_code'])
inventor_locations = inventor_locations.merge(families, on='appln_id')
inventor_locations['year'] = inventor_locations.filing_date.astype(str).str[:4]


In [ ]:
country_families = (inventor_locations.groupby(['docdb_family_id', 'ctry_code', 'year'])
                    .size().rename('num_addr').reset_index())
country_families['cntryshare'] = country_families.num_addr / country_families.groupby(
    ['docdb_family_id', 'year']).num_addr.transform('sum')
country_cpc = country_families.merge(family_cpc, on='docdb_family_id')
country_cpc['wt'] = country_cpc.cntryshare * country_cpc.ratio
country_cpc.to_parquet(PATSTAT / 'cntry_inv_patfam_cpcall.parquet', index=False)
country_cpc.head()


In [ ]:
world_families = country_cpc.docdb_family_id.nunique()
pd.Series({'world_families_with_country_and_cpc': world_families,
           'countries': country_cpc.ctry_code.nunique(),
           'family_country_code_year_rows': len(country_cpc)})


`6_viz_firm_cities.ipynb` reads this country table and counts distinct families for country/CPC and country/GBB maps. The summary above gives the number of families with both country and CPC information.
